**Классификация комментариев**

Магазину нужен инструмент, который будет искать токсичные комментарии и отправлять их на модерацию.\
**Задача:** подобрать и обучить модель, которая будет классифицировать комментарии на позитивные и негативные.

**Ход исследования:**
1. Загрузка данных
2. Подготовка данных
3. Обучение моделей
4. Общий вывод

In [1]:
!pip install pymystem3 -q

In [2]:
import numpy as np
import pandas as pd
from pymystem3 import Mystem
import re
import nltk
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
from time import time
from nltk.corpus import stopwords as nltk_stopwords
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV
)
from sklearn.feature_extraction.text import TfidfVectorizer

[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/jovyan/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


## Загрузка данных

In [3]:
# чтение файла с данными и сохранение в переменную 
try:
    data = pd.read_csv('toxic_comments.csv', index_col=[0])
except:
    data = pd.read_csv('https://code.s3.yandex.net/datasets/toxic_comments.csv', index_col=[0])

In [4]:
# получение первых строк датафрейма
data.head(3)

,text,toxic
0,Explanation\nWhy the edits made under my usern...,0
1,D'aww! He matches this background colour I'm s...,0
2,"Hey man, I'm really not trying to edit war. It...",0


In [5]:
# получение общей информации
data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 159292 entries, 0 to 159450
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   text    159292 non-null  object
 1   toxic   159292 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 3.6+ MB


In [6]:
# подсчёт пропущенных значений
data.isna().sum()

text     0
toxic    0
dtype: int64

In [7]:
# проверка на явные дубликаты
data.duplicated().sum(), data.index.duplicated().sum()

(0, 0)

In [8]:
data.columns

Index(['text', 'toxic'], dtype='object')

**Вывод**

Загрузили данные. Дубликатов и пропусков нет.

## Подготовка данных

Лемматизируем и очистим текст комментариев, оставим только английский символы и пробелы, приведем к нижнему регистру.

In [9]:
wnl = WordNetLemmatizer() # создание класса для лемматизации

In [10]:
# функция для определения тега POS ("part-of-speech") для слов текста
def get_wordnet_pos(word):
    tag = nltk.pos_tag([word])[0][1][0].upper()
    tag_dict = {"J": wordnet.ADJ,
                "N": wordnet.NOUN,
                "V": wordnet.VERB,
                "R": wordnet.ADV}
    return tag_dict.get(tag, wordnet.NOUN)

In [14]:
def lemmatize(text):
    text = [wnl.lemmatize(word, get_wordnet_pos(word)) for word in nltk.word_tokenize(text)]
    return " ".join(text)

display(data.head())

def clear_text(text):
    text = re.sub(r'[^a-zA-Z ]', ' ', text).lower() # очищаем от всех символов, кроме пробела
    clear_text = " ".join(text.split()) # объединяем в строку через пробел(получаем список без пробелов)
    return clear_text

print(clear_text(data['text'].iloc[0]))
# применяем к нашему столбцу с комментариями
data['lemm_text'] = data['text'].apply(lambda x: lemmatize(clear_text(x)))

data.head()

,text,toxic
0,Explanation\nWhy the edits made under my usern...,0
1,D'aww! He matches this background colour I'm s...,0
2,"Hey man, I'm really not trying to edit war. It...",0
3,"""\nMore\nI can't make any real suggestions on ...",0
4,"You, sir, are my hero. Any chance you remember...",0


explanation why the edits made under my username hardcore metallica fan were reverted they weren t vandalisms just closure on some gas after i voted at new york dolls fac and please don t remove the template from the talk page since i m retired now


,text,toxic,lemm_text
0,Explanation\nWhy the edits made under my usern...,0,explanation why the edits make under my userna...
1,D'aww! He matches this background colour I'm s...,0,d aww he match this background colour i m seem...
2,"Hey man, I'm really not trying to edit war. It...",0,hey man i m really not try to edit war it s ju...
3,"""\nMore\nI can't make any real suggestions on ...",0,more i can t make any real suggestion on impro...
4,"You, sir, are my hero. Any chance you remember...",0,you sir be my hero any chance you remember wha...


In [19]:
# посчитаем сколько раз встречается каждый класс
print(data['toxic'].value_counts())

0    143106
1     16186
Name: toxic, dtype: int64


Получили дисбаланс классов в целевом признаке.

Разделим данные на тренировочную и тестовую выборки, сделав стратификацию по целевому признаку.

In [20]:
# сохранение входного и целевого признаков
features = data['lemm_text']
target = data['toxic']

# разделение на тренировочную и тестовую выборки    
features_train, features_test, target_train, target_test = \
train_test_split(features, target, test_size=0.25, random_state=42, stratify=target)

Загрузим стоп-слова, применим векторизацию.

In [21]:
nltk.download('stopwords')
stopwords = list(nltk_stopwords.words('english'))
count_tf_idf = TfidfVectorizer(stop_words=stopwords) # создание счётчика с указанием в нём стоп-слов
# подсчёт TF-IDF для обучающей выборки
features_train = count_tf_idf.fit_transform(features_train)
features_test = count_tf_idf.transform(features_test)
print(features_train.shape)
print(features_test.shape)

[nltk_data] Downloading package stopwords to /home/jovyan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


(119469, 127210)
(39823, 127210)


**Вывод**

Лемматизировали и очистили тексты. Применили векторизацию. Подготовили тренировочную и тестовую выборки

## Обучение

Создадим функцию обучения модели, с подбором гиерпареметров

In [22]:
results = pd.DataFrame(columns=['model', 'best_score', 'fit_time', 'pred_time'])

In [23]:
def model(estimator, param_grid, features_train, features_test, target_train, target_test):
    model = GridSearchCV(estimator=estimator,
                         param_grid=param_grid,
                         n_jobs=-1,
                         cv=3,
                         scoring='f1')

    model.fit(features_train, target_train)
    best_score = model.best_score_ 
    print(f"Лучшая скорость на кросс-валидации {best_score}")
    # получаем лучшую модель 
    best_model = model.best_estimator_
    print(f"Лучшая модель и её параметры:\n\n {best_model}")
    # получаем предсказания на обучающем наборе данных
    y_pred = best_model.predict(features_train)
    # замеряем время обучения (в секундах)
    start = time()
    best_model.fit(features_train, target_train)
    end = time()
    fit_time = end - start
    # замеряем время предсказания (в секундах)
    start = time()
    pred_time = best_model.predict(features_train)
    end = time()
    pred_time = end - start
    return best_model, best_score, fit_time, pred_time

In [24]:
best_model_lr, best_score_lr, fit_time_lr, pred_time_lr = model(LogisticRegression(random_state=42,
                                                                                   solver='liblinear',
                                                                                   class_weight='balanced'),
                                                     {'C': [1,10]},
                                                     features_train, features_test, target_train, target_test)
results.loc[0] = ['LogisticRegression', best_score_lr, fit_time_lr, pred_time_lr]
results

Лучшая скорость на кросс-валидации 0.7582625611818519
Лучшая модель и её параметры:

 LogisticRegression(C=10, class_weight='balanced', random_state=42,
                   solver='liblinear')


,model,best_score,fit_time,pred_time
0,LogisticRegression,0.758263,25.480026,0.012243


In [25]:
best_model_d, best_score_d, fit_time_d, pred_time_d = model(DecisionTreeClassifier(class_weight='balanced', random_state=42),
                                                     {'max_depth':[20, 40]},
                                                     features_train, features_test, target_train, target_test)
results.loc[1] = ['DecisionTreeClassifier', best_score_d, fit_time_d, pred_time_d]
results

Лучшая скорость на кросс-валидации 0.641969466549833
Лучшая модель и её параметры:

 DecisionTreeClassifier(class_weight='balanced', max_depth=40, random_state=42)


,model,best_score,fit_time,pred_time
0,LogisticRegression,0.758263,25.480026,0.012243
1,DecisionTreeClassifier,0.641969,39.596949,0.037807


In [27]:
pred_test_lr = best_model_lr.predict(features_test)
f1_lr_pred = f1_score(target_test, pred_test_lr)
f1_lr_pred

0.756961155036095

## Общий вывод

В ходе работы загрузили данные, проверили на отсутсвие дубликатов и пропусков. 

Отметили дисбаланс классов. Лемматизировали и очистили тексты. Применили векторизацию. 

Подготовили тренировочную и тестовую выборки.
Для задачи бинарной классификации обучили модели LogisticRegression, DecisionTreeClassifier. 

Лучшей оказалась LogisticRegression с параметрами (C=10, class_weight='balanced', random_state=42, solver='liblinear') и значением метрики качества на тестовой выборке f1 = 0,76 (по требованиям заказчика не меньше 0,75).